# How to use the oneClick-ML utility in the Atlas cluster

## Import `digitalhub` library and create a (or get an existing) project

In [ ]:
import digitalhub as dh

The following method works both if the project has already been created or not

In [ ]:
project = dh.get_or_create_project('oneclick-ml-atlas')

## Load input dataset

A file that is ready to be used, in the Digitalhub terminology, is an artifact. After that, it can be found on the "Data items" and "Artifact" section of the Core managment UI.

The two files in this example are `data.csv` and `config.ini`. The following cell shows how to load the first file directly from the workspace space.

In [ ]:
csv_data = project.log_artifact(
    name='csv_input_data' ,
    kind='artifact' ,
    source='oneClick-ML/data.csv'
)
csv_data.key

This cell shows how to fetch it if it's already been created (for example if it's not the first time you run the notebook cells).

In [ ]:
csv_data = project.get_artifact('csv_input_data')
csv_data.key

The same can be done for the second file:

In [ ]:
config_data = project.log_artifact(
    name='config_ini_file' ,
    kind='artifact' ,
    source='oneClick-ML/config.ini'
)
config_data.key

In [ ]:
config_data = project.get_artifact('config_ini_file')
config_data.key

## General overview on how the functions work

A function, in the Digitalhub terminology, is a chunk of code that has a specific purpose. For a multi-step pipeline, divide every independent step declaring a function for it. The advantage of this is that you can easily re-run a code with different inputs.

A function first needs to be declared, then needs to be builded, and after these steps can be executed ("runned") on different data multiple times.

If it's the first time that the function is used, first register it by running `Cell 1`, then build it by running `Cell 3` and then execute it by running `Cell 5`. It can be re-executed on different data by changing input and output data in the options `input` and `parameters`.

If the function has already been declared and builded, and:
* the notebook has been reopened (= the jupyter kernel has been restarted): rerun the first cells, and after that run `Cell 2` and `Cell 5`
* the code has been modified: run `Cell 2` to get the function, then `Cell 4` to fetch the build and `Cell 1` ***with the parameter `image` decommented***; finally run directly `Cell 5`

## Declaration of the main helper function

In [ ]:
# Cell 1 - declares a function

main_func = project.new_function(
    name='main_function',
    kind='python',
    python_version='PYTHON3_10',
    requirements='oneClick-ML/requirements.txt',
    code_src='oneClick-ML', # this is the folder where all the original repo files are cloned
    handler='main:helper_main', # main is the original main.py from the repo : train is the new function (that calls the original main function)
    # image=main_func_ref,
)

If it has already been declared, just get it. It's possible to retrieve multiple versions of a function by pasting the corresponding id (Core managment UI > Functions > Click on functions > id).

In [ ]:
# Cell 2 - retrieves an already declared function

main_func = project.get_function('main_function',
    # entity_id=''
)

## Building of the main helper function

In [ ]:
# Cell 3 - builds the function

main_func_build = main_func.run(action='build',
    wait=True,
    resources={'cpu': '4', 'mem': '8Gi', 'disk': '10Gi'},
)

In [ ]:
# Cell 4 - gets the reference of the function

main_func_ref = main_func.refresh().spec.image
print(main_func_ref)

## Running of the main helper function

The execution can be monitored in the "Runs" section of the Core managment UI, while the result can be found in the "Artifacts" section.

In [ ]:
# Cell 5 - runs a function

main_func_run = main_func.run(action='job',
    inputs={
        'csv_file': csv_data.key ,
        'config_file': config_data.key
    },
    parameters={
        # Place here the name of the output artifact
        'output_artifact_name': 'output_data' ,
    },
    resources={
        'cpu': '4',
        'mem': '8Gi',
        'disk': '20Gi',
    },
    local_execution=False,
)